# Hunyuan3D-2 Paint (v5, free Colab)

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. Under cell 1 click **Choose files** → `s2f.zip` (Desktop → `hunyuan_paint`). Then you can walk away.
4. ~15–20 min later `painted.glb` downloads by itself.

No restarts needed. Every step prints the RAM in use; if something stops, copy the last lines of that cell.

In [ ]:
# 1. upload the job zip first (mesh + 3 photos, prepared on the Mac), then you can walk away
import zipfile, os
from google.colab import files
up = files.upload()
zipfile.ZipFile(next(iter(up))).extractall('/content/job')
print(sorted(os.listdir('/content/job')))

In [ ]:
# 2. install (~6-8 min). numpy and Pillow are pinned to the versions already loaded in this session, so nothing
# changes under the running Python and no restart is needed
%cd /content
![ -d Hunyuan3D-2 ] || git clone -q https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
%cd /content/Hunyuan3D-2
!pip freeze | grep -iE "^(numpy|pillow)==" > /content/keep.txt; cat /content/keep.txt
!pip -q install -r requirements.txt -c /content/keep.txt 2>&1 | grep -v "dependency resolver\|incompatible" || true
!pip -q install -e . --no-deps
!cd hy3dgen/texgen/custom_rasterizer && pip -q install . --no-build-isolation -c /content/keep.txt && echo custom_rasterizer OK
!cd hy3dgen/texgen/differentiable_renderer && pip -q install . --no-build-isolation -c /content/keep.txt && echo differentiable_renderer OK
!(fallocate -l 16G /swapfile && chmod 600 /swapfile && mkswap /swapfile >/dev/null && swapon /swapfile && echo swap ON) 2>/dev/null || echo "swap not allowed (ok)"
import numpy, PIL; print('numpy', numpy.__version__, 'Pillow', PIL.__version__, '(unchanged)')

In [ ]:
# 3. download only the weights that are used (skips the 7.3 GB fp32 copy), ~3 min
import psutil
from huggingface_hub import snapshot_download
ROOT = snapshot_download('tencent/Hunyuan3D-2',
                         allow_patterns=['hunyuan3d-delight-v2-0/*', 'hunyuan3d-paint-v2-0-turbo/*'],
                         ignore_patterns=['*unet/diffusion_pytorch_model.bin'])
print('weights in', ROOT)
ram = lambda tag: print('%-28s RAM %.1f GB used' % (tag, psutil.virtual_memory().used / 1e9))
ram('after download')

In [ ]:
# 4. load the models one by one straight onto the GPU, so the 12 GB of RAM never holds them all
import sys, gc, json, time
sys.path.insert(0, '/content/Hunyuan3D-2')
os.chdir('/content/Hunyuan3D-2')
import torch
from safetensors.torch import load_file
from diffusers import AutoencoderKL, LCMScheduler, DDIMScheduler
from transformers import CLIPTextModel, CLIPTokenizer, CLIPImageProcessor
import hy3dgen.texgen.pipelines as P
from hy3dgen.texgen.utils.multiview_utils import Multiview_Diffusion_Net
from hy3dgen.texgen.hunyuanpaint.pipeline import HunyuanPaintPipeline
from hy3dgen.texgen.hunyuanpaint.unet.modules import UNet2p5DConditionModel
from diffusers.models import UNet2DConditionModel
print('GPU:', torch.cuda.get_device_name(0)); ram('start')
MV = ROOT + '/hunyuan3d-paint-v2-0-turbo'
cfg = P.Hunyuan3DTexGenConfig(ROOT + '/hunyuan3d-delight-v2-0', MV, 'hunyuan3d-paint-v2-0-turbo')

paint = P.Hunyuan3DPaintPipeline.__new__(P.Hunyuan3DPaintPipeline)
paint.config = cfg
paint.models = {}
paint.render = P.MeshRender(default_resolution=cfg.render_size, texture_size=cfg.texture_size)

paint.models['delight_model'] = P.Light_Shadow_Remover(cfg)  # ~2 GB, moves itself to the GPU
gc.collect(); ram('delight model on GPU')

# paint UNet: built in fp16 directly on the GPU, filled from the 3.7 GB fp16 file read straight into GPU memory
with torch.device('cuda'):
    torch.set_default_dtype(torch.float16)
    unet = UNet2p5DConditionModel(UNet2DConditionModel.from_config(json.load(open(MV + '/unet/config.json'))))
    torch.set_default_dtype(torch.float32)
sd = load_file(MV + '/unet/diffusion_pytorch_model.safetensors', device='cuda')
missing, extra = unet.load_state_dict(sd, strict=False)
assert not missing, missing[:5]  # the extra keys are unused image-prompt layers
del sd; torch.cuda.empty_cache(); gc.collect(); ram('paint UNet on GPU')

f16 = torch.float16
vae = AutoencoderKL.from_pretrained(MV + '/vae', torch_dtype=f16).to('cuda')
text = CLIPTextModel.from_pretrained(MV + '/text_encoder', torch_dtype=f16, low_cpu_mem_usage=True).to('cuda')
gc.collect(); ram('vae + text encoder on GPU')
pipe = HunyuanPaintPipeline(vae=vae, text_encoder=text, tokenizer=CLIPTokenizer.from_pretrained(MV + '/tokenizer'),
                            unet=unet, scheduler=DDIMScheduler.from_pretrained(MV + '/scheduler'),
                            feature_extractor=CLIPImageProcessor.from_pretrained(MV + '/feature_extractor'))
pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config, timestep_spacing='trailing')
pipe.set_turbo(True)
pipe.set_progress_bar_config(disable=True)
mv = Multiview_Diffusion_Net.__new__(Multiview_Diffusion_Net)
mv.device, mv.view_size, mv.pipeline = 'cuda', 512, pipe
paint.models['multiview_model'] = mv
ram('all models ready')
print('GPU memory %.1f GB' % (torch.cuda.memory_allocated() / 1e9))

In [ ]:
# 5. paint (~5-10 min) and download
import trimesh
from PIL import Image
P.mesh_uv_wrap = lambda m: m  # unwrapped on the Mac already
t = time.time()
mesh = trimesh.load('/content/job/mesh.glb', force='mesh')
imgs = [Image.open('/content/job/' + n) for n in ['front.png', 'left.png', 'back.png']]
print('mesh', len(mesh.faces), 'faces, photos', len(imgs)); ram('before painting')
paint(mesh, image=imgs).export('/content/painted.glb')
print('painted in %.0f s' % (time.time() - t)); ram('done')
files.download('/content/painted.glb')